# STEP3-00. 배포용 파일 준비 (코랩에서 1번만 실행)

내 컴퓨터에서 대시보드를 실행하려면 **데이터 · 모델 · 설치 목록** 3가지가 필요합니다.
이 노트북은 3가지를 한 폴더에 모아 **zip 파일 하나로 내려받게** 해 줍니다.

| 만드는 파일 | 내용 |
|---|---|
| `data/orders.parquet` | 분석용 주문 데이터 (주문 1건 = 1행) |
| `data/items.parquet` | 분석용 상품 데이터 (카테고리 분석용) |
| `data/seller_table.csv` | 판매자 표 (판매자 이탈 화면용, 10 노트북에서 저장) |
| `data/olist_facts.csv` · `olist_model_guide.md` · `olist_seller_policy.pdf` | 분석 Q&A 챗봇의 자료 (`Olist/rag` 폴더에서 복사) |
| `model/` | STEP1에서 저장한 모델 4개 (복사) |
| `requirements.txt` | 설치할 패키지 목록 (**코랩과 같은 버전**으로 자동 작성) |

## 준비
- STEP1 노트북 06 · 07_1 · 08 · 09 실행 → `model` 폴더에 pkl 4개
- 10 노트북 끝의 **판매자 표 저장 셀** 실행 → `model/seller_table.csv`

## 흐름
```
① Drive 연결 → ② 데이터 만들기 → ③ 모델 · 판매자 표 복사 → ④ requirements.txt 만들기 → ⑤ zip 내려받기
```

In [1]:
# 셀 1. Drive 연결 + 폴더 만들기
from google.colab import drive
drive.mount('/content/drive')

import os
base_path   = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
deploy_path = "/content/deploy_files"                 # 내려받을 파일을 모을 폴더
os.makedirs(f"{deploy_path}/data", exist_ok=True)
os.makedirs(f"{deploy_path}/model", exist_ok=True)

Mounted at /content/drive


## ② 데이터 만들기

PART 5와 같은 기준(배송 완료 · 2017-01-05 ~ 2018-08-29)으로 만듭니다.
62MB CSV 대신 **필요한 열만** 저장하므로 파일이 작아지고, 화면도 빨리 열립니다.

In [2]:
# 셀 2. 분석용 데이터 만들기 → parquet 저장
import pandas as pd

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])

pay = df.drop_duplicates(["order_id", "payment_sequential"]).groupby("order_id")["payment_value"].sum()

orders = df.drop_duplicates("order_id")                                    # 주문 1건 = 1행
orders = orders[(orders["order_status"] == "delivered")
                & orders["order_delivered_customer_date"].notna()
                & (orders["order_purchase_timestamp"] >= "2017-01-05")
                & (orders["order_purchase_timestamp"] < "2018-08-30")].copy()
orders["주문시각"] = orders["order_purchase_timestamp"]
orders["매출"] = orders["order_id"].map(pay)
orders["배송일수"] = (orders["order_delivered_customer_date"] - orders["주문시각"]).dt.total_seconds() / 86400
orders["지연"] = orders["order_delivered_customer_date"] > orders["order_estimated_delivery_date"]
orders = orders[["order_id", "customer_unique_id", "customer_state", "주문시각", "매출", "배송일수", "지연", "review_score"]]

items = df.dropna(subset=["order_item_id"]).drop_duplicates(["order_id", "order_item_id"])  # 상품 1개 = 1행
items = items[items["order_id"].isin(orders["order_id"])][["order_id", "product_category_name_english", "price"]]

orders.to_parquet(f"{deploy_path}/data/orders.parquet", index=False)
items.to_parquet(f"{deploy_path}/data/items.parquet", index=False)
print(f"주문 {len(orders):,}건 · 매출 R$ {orders['매출'].sum() / 1e6:.2f}M   (PART 5 : 96,203건 · R$ 15.37M)")

주문 96,203건 · 매출 R$ 15.37M   (PART 5 : 96,203건 · R$ 15.37M)


In [3]:
# 셀 3. 모델 4개 + 판매자 표 복사
import shutil

models = ["part6_recommend.pkl",          # 06 상품 추천
          "part7_1_sentiment_ml.pkl",     # 07_1 리뷰 감성 (머신러닝)
          "part8_seller_segment.pkl",     # 08 판매자 유형
          "part9_seller_churn.pkl"]       # 09 판매자 이탈
for name in models:
    shutil.copy(f"{base_path}/model/{name}", f"{deploy_path}/model/{name}")

table = f"{base_path}/model/seller_table.csv"
if os.path.exists(table):
    shutil.copy(table, f"{deploy_path}/data/seller_table.csv")
else:
    print("⚠️ seller_table.csv가 없습니다. 10 노트북 끝의 '판매자 표 저장' 셀을 먼저 실행하세요.")

# STEP4 분석 Q&A 챗봇용 자료 (사실표 · 모델 설명서 · 규정집 PDF)
for name in ["olist_facts.csv", "olist_model_guide.md", "olist_seller_policy.pdf"]:
    if os.path.exists(f"{base_path}/rag/{name}"):
        shutil.copy(f"{base_path}/rag/{name}", f"{deploy_path}/data/{name}")

print("model :", sorted(os.listdir(f"{deploy_path}/model")))
print("data  :", sorted(os.listdir(f"{deploy_path}/data")))

model : ['part6_recommend.pkl', 'part7_1_sentiment_ml.pkl', 'part8_seller_segment.pkl', 'part9_seller_churn.pkl']
data  : ['items.parquet', 'olist_facts.csv', 'olist_model_guide.md', 'orders.parquet', 'seller_table.csv']


## ④ requirements.txt 만들기 — ⭐ 배포에서 가장 중요한 단계

모델 파일(`.pkl`)은 **만든 프로그램 버전**에 맞춰 저장됩니다.
내 컴퓨터 · 배포 서버의 버전이 다르면 모델이 열리지 않거나 **결과가 달라질 수 있습니다.**
그래서 모델을 학습한 **코랩의 버전을 그대로 적어** 둡니다.

| 패키지 | 버전 고정 | 이유 |
|---|---|---|
| scikit-learn | ✅ | 추천 KNN · TF-IDF · K-Means · 로지스틱 회귀 `.pkl` |
| numpy | ✅ | `.pkl` 안의 숫자 배열 |
| streamlit | 1.40 이상 | 메뉴 묶음(`st.navigation`)을 쓰려면 필요 |
| pandas · pyarrow · plotly · joblib | 최신 | 데이터 · 그래프 도구 |
| langchain-core · langchain-openai · langchain-community · pypdf | 최신 | 분석 Q&A 챗봇 (RAG, PDF 읽기) |

> 딥러닝(TensorFlow)은 넣지 않습니다. 설치 파일이 수백 MB라 내 컴퓨터 설치와 배포가 오래 걸립니다.

In [4]:
# 셀 4. requirements.txt 만들기 (코랩 버전 그대로)
import sys, numpy, sklearn

requirements = f"""streamlit>=1.40
pandas
pyarrow
plotly
joblib
numpy=={numpy.__version__}
scikit-learn=={sklearn.__version__}
langchain-core
langchain-openai
langchain-community
pypdf
"""
with open(f"{deploy_path}/requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)
print("⭐ 파이썬 버전 :", sys.version.split()[0], "← 내 컴퓨터 · 배포 서버도 이 버전(앞 두 자리)으로 맞춥니다.")

streamlit>=1.40
pandas
pyarrow
plotly
joblib
numpy==2.4.6
scikit-learn==1.6.1
langchain-core
langchain-openai
langchain-community
pypdf

⭐ 파이썬 버전 : 3.13.18 ← 내 컴퓨터 · 배포 서버도 이 버전(앞 두 자리)으로 맞춥니다.


In [5]:
# 셀 5. zip으로 묶어 내려받기
shutil.make_archive("/content/olist_deploy_files", "zip", deploy_path)

from google.colab import files
files.download("/content/olist_deploy_files.zip")    # 브라우저 다운로드 폴더에 저장됩니다.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 다음 할 일

1. 내려받은 `olist_deploy_files.zip`의 압축을 풉니다.
2. 안에 있는 `data` · `model` 폴더와 `requirements.txt`를 **`olist_dashboard` 폴더 안에 복사**합니다.

```
olist_dashboard/
 ├─ app.py · views/ · utils/          ← 대시보드 코드
 ├─ data/                              ← 여기에 복사 (orders · items · seller_table)
 ├─ model/                             ← 여기에 복사 (pkl 4개)
 └─ requirements.txt                   ← 여기에 복사 (원래 파일을 덮어씀)
```